In [1]:

from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data/raw/railway_timetable")

# List the files so we use the actual downloaded filename
for file in DATA_DIR.iterdir():
    if file.is_file():
        print(file.name)


Train_details_22122017.csv


In [2]:

TIMETABLE_PATH = DATA_DIR / "Train_details_22122017.csv"

timetable = pd.read_csv(
    TIMETABLE_PATH,
    dtype={
        "Train No": "string",
        "Station Code": "string",
        "Source Station": "string",
        "Destination Station": "string",
    },
    low_memory=False,
)

# Standardize column names and remove surrounding whitespace
timetable.columns = timetable.columns.str.strip()

for col in timetable.select_dtypes(include=["object", "string"]).columns:
    timetable[col] = timetable[col].str.strip()

# Convert fields to appropriate types
timetable["SEQ"] = pd.to_numeric(timetable["SEQ"], errors="coerce")
timetable["Distance"] = pd.to_numeric(
    timetable["Distance"], errors="coerce"
)

print("Shape:", timetable.shape)
print("Missing values:\n", timetable.isna().sum())
print("Sequence type:", timetable["SEQ"].dtype)
print("Distance type:", timetable["Distance"].dtype)


Shape: (186124, 12)
Missing values:
 Train No                     0
Train Name                   0
SEQ                          5
Station Code                 0
Station Name                 0
Arrival time                 5
Departure Time               5
Distance                    10
Source Station              10
Source Station Name         10
Destination Station         10
Destination Station Name    10
dtype: int64
Sequence type: float64
Distance type: float64


In [3]:

# 1. Inspect rows with missing values
missing_rows = timetable[timetable.isna().any(axis=1)]

print("Rows with at least one missing value:", len(missing_rows))
display(missing_rows.head(15))

# 2. Check whether station sequence numbers are valid
print("\nSequence values:")
print(timetable["SEQ"].describe())

print("\nRows with missing or invalid sequence numbers:")
display(timetable[
    timetable["SEQ"].isna() | (timetable["SEQ"] < 1)
].head(15))

# 3. Check distances within each train route
route_check = timetable.sort_values(
    ["Train No", "SEQ"]
).copy()

route_check["distance_change"] = (
    route_check.groupby("Train No")["Distance"].diff()
)

print("\nNegative distance changes:", 
      (route_check["distance_change"] < 0).sum())

print("\nExample route after sorting:")
display(
    route_check[route_check["Train No"] == "107"][
        ["Train No", "SEQ", "Station Code",
         "Arrival time", "Departure Time",
         "Distance", "distance_change"]
    ]
)


Rows with at least one missing value: 10


,Train No,Train Name,SEQ,Station Code,Station Name,Arrival time,Departure Time,Distance,Source Station,Source Station Name,Destination Station,Destination Station Name
2308,6919,UBL-BJP SPL,33.0,KGIH,KALGURKI H,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN
2309,K,12:16:00,NaN,214,UBL,HUBLI JN.,BJP,NaN,<NA>,NaN,<NA>,NaN
2320,6920,BJP-UBL SPL,6.0,KGIH,KALGURKI H,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN
2321,K,14:15:00,NaN,33,BJP,BIJAPUR JN,UBL,NaN,<NA>,NaN,<NA>,NaN
135826,56905,BJP-UBL PASS,14.0,KGIH,KALGURKI H,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN
135827,K,07:46:00,NaN,142,SUR,SOLAPUR,UBL,NaN,<NA>,NaN,<NA>,NaN
135884,56906,UBL-BJP PASS,29.0,KGIH,KALGURKI H,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN
135885,K,18:46:00,NaN,214,UBL,HUBLI JN.,SUR,NaN,<NA>,NaN,<NA>,NaN
140468,57641,BGK PASS,19.0,KGIH,KALGURKI H,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN
140469,K,14:54:00,NaN,142,SUR,SOLAPUR,GDG,NaN,<NA>,NaN,<NA>,NaN



Sequence values:
count    186119.000000
mean         13.916124
std          12.778720
min           1.000000
25%           5.000000
50%          11.000000
75%          19.000000
max         118.000000
Name: SEQ, dtype: float64

Rows with missing or invalid sequence numbers:


,Train No,Train Name,SEQ,Station Code,Station Name,Arrival time,Departure Time,Distance,Source Station,Source Station Name,Destination Station,Destination Station Name
2309,K,12:16:00,NaN,214,UBL,HUBLI JN.,BJP,NaN,<NA>,NaN,<NA>,NaN
2321,K,14:15:00,NaN,33,BJP,BIJAPUR JN,UBL,NaN,<NA>,NaN,<NA>,NaN
135827,K,07:46:00,NaN,142,SUR,SOLAPUR,UBL,NaN,<NA>,NaN,<NA>,NaN
135885,K,18:46:00,NaN,214,UBL,HUBLI JN.,SUR,NaN,<NA>,NaN,<NA>,NaN
140469,K,14:54:00,NaN,142,SUR,SOLAPUR,GDG,NaN,<NA>,NaN,<NA>,NaN



Negative distance changes: 0

Example route after sorting:


,Train No,SEQ,Station Code,Arrival time,Departure Time,Distance,distance_change
0,107,1.0,SWV,00:00:00,10:25:00,0.0,NaN
1,107,2.0,THVM,11:06:00,11:08:00,32.0,32.0
2,107,3.0,KRMI,11:28:00,11:30:00,49.0,17.0
3,107,4.0,MAO,12:10:00,00:00:00,78.0,29.0


In [4]:

# Show all rows where SEQ is missing
bad_rows = timetable[timetable["SEQ"].isna()]

print("Rows with missing SEQ:", len(bad_rows))

display(bad_rows[
    ["Train No", "Train Name", "SEQ", "Station Code",
     "Station Name", "Arrival time", "Departure Time",
     "Distance", "Source Station", "Destination Station"]
])

# Inspect the original CSV lines around the first suspicious row
raw_path = TIMETABLE_PATH

with open(raw_path, "r", encoding="utf-8-sig", errors="replace") as f:
    lines = f.readlines()

print("\nOriginal CSV header:")
print(lines[0].rstrip())

print("\nOriginal CSV lines around the first suspicious row:")
for line_number in range(2307, 2312):
    print(f"Line {line_number + 1}: {lines[line_number].rstrip()}")


Rows with missing SEQ: 5


,Train No,Train Name,SEQ,Station Code,Station Name,Arrival time,Departure Time,Distance,Source Station,Destination Station
2309,K,12:16:00,NaN,214,UBL,HUBLI JN.,BJP,NaN,<NA>,<NA>
2321,K,14:15:00,NaN,33,BJP,BIJAPUR JN,UBL,NaN,<NA>,<NA>
135827,K,07:46:00,NaN,142,SUR,SOLAPUR,UBL,NaN,<NA>,<NA>
135885,K,18:46:00,NaN,214,UBL,HUBLI JN.,SUR,NaN,<NA>,<NA>
140469,K,14:54:00,NaN,142,SUR,SOLAPUR,GDG,NaN,<NA>,<NA>



Original CSV header:
Train No,Train Name,SEQ,Station Code,Station Name,Arrival time,Departure Time,Distance,Source Station,Source Station Name,Destination Station,Destination Station Name

Original CSV lines around the first suspicious row:
Line 2308: 6919,UBL-BJP SPL,31,BSRX,BASAVANA BAG,12:07:00,12:08:00,203,UBL,HUBLI JN.,BJP,BIJAPUR JN
Line 2309: 6919,UBL-BJP SPL,32,KDGI,KUDGI H,12:12:00,12:13:00,208,UBL,HUBLI JN.,BJP,BIJAPUR JN
Line 2310: 6919,UBL-BJP SPL,33,KGIH,KALGURKI H,NA,NA,NA,NA,NA,NA,NA
Line 2311: K,12:16:00,12:17:00,214,UBL,HUBLI JN.,BJP,BIJAPUR JN,NA,NA,NA,NA
Line 2312: 6919,UBL-BJP SPL,34,MVD,MULVAD,12:22:00,12:23:00,222,UBL,HUBLI JN.,BJP,BIJAPUR JN


In [5]:

# Keep the original loaded dataframe unchanged
timetable_clean = timetable.copy()

before = len(timetable_clean)

# Remove rows that cannot reliably define a train's route
required_columns = [
    "Train No",
    "SEQ",
    "Station Code",
    "Distance",
]

timetable_clean = timetable_clean.dropna(
    subset=required_columns
).copy()

# Sequence and distance should be valid positive/non-negative numbers
timetable_clean = timetable_clean[
    (timetable_clean["SEQ"] >= 1) &
    (timetable_clean["Distance"] >= 0)
].copy()

print("Rows before cleaning:", before)
print("Rows after cleaning:", len(timetable_clean))
print("Rows removed:", before - len(timetable_clean))
print(
    "Remaining missing required values:",
    timetable_clean[required_columns].isna().sum().sum()
)


Rows before cleaning: 186124
Rows after cleaning: 186114
Rows removed: 10
Remaining missing required values: 0


In [6]:

from pathlib import Path

OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "railway_timetable_clean.csv"

timetable_clean.to_csv(OUTPUT_PATH, index=False)

print("Saved to:", OUTPUT_PATH.resolve())


Saved to: C:\Users\Harsh\Desktop\kalantar\data\processed\railway_timetable_clean.csv


In [7]:

# Sort stations in their scheduled order within each train
routes = timetable_clean.sort_values(
    ["Train No", "SEQ"]
).copy()

# Identify the next station on each train's route
routes["Next Station Code"] = (
    routes.groupby("Train No")["Station Code"].shift(-1)
)

routes["Next Station Name"] = (
    routes.groupby("Train No")["Station Name"].shift(-1)
)

routes["Next Distance"] = (
    routes.groupby("Train No")["Distance"].shift(-1)
)

# Each row with a next station represents a route segment
edges = routes.dropna(
    subset=["Next Station Code", "Next Distance"]
).copy()

edges["Segment Distance"] = (
    edges["Next Distance"] - edges["Distance"]
)

# Inspect any invalid segments before saving
print("Total route segments:", len(edges))
print(
    "Negative segment distances:",
    (edges["Segment Distance"] < 0).sum()
)

display(edges[
    ["Train No", "Station Code", "Next Station Code",
     "Distance", "Next Distance", "Segment Distance"]
].head(10))


Total route segments: 175002
Negative segment distances: 0


,Train No,Station Code,Next Station Code,Distance,Next Distance,Segment Distance
4354,10103,CSMT,DR,0.0,8.0,8.0
4355,10103,DR,TNA,8.0,33.0,25.0
4356,10103,TNA,PNVL,33.0,70.0,37.0
4357,10103,PNVL,MNI,70.0,177.0,107.0
4358,10103,MNI,KHED,177.0,245.0,68.0
4359,10103,KHED,CHI,245.0,275.0,30.0
4360,10103,CHI,SGR,275.0,317.0,42.0
4361,10103,SGR,RN,317.0,351.0,34.0
4362,10103,RN,ADVI,351.0,382.0,31.0
4363,10103,ADVI,RAJP,382.0,414.0,32.0


In [8]:

# Check route segment quality
print("Total segments:", len(edges))

print(
    "Zero-distance segments:",
    (edges["Segment Distance"] == 0).sum()
)

print(
    "Segments over 300 km:",
    (edges["Segment Distance"] > 300).sum()
)

print(
    "Missing segment distances:",
    edges["Segment Distance"].isna().sum()
)

# Check duplicate segments for the same train
duplicate_segments = edges.duplicated(
    subset=["Train No", "Station Code", "Next Station Code"]
).sum()

print("Duplicate train segments:", duplicate_segments)

# Inspect the largest segment distances
display(
    edges.nlargest(10, "Segment Distance")[
        ["Train No", "Station Code", "Next Station Code",
         "Segment Distance"]
    ]
)

# Save the route network
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EDGE_PATH = OUTPUT_DIR / "railway_network_edges.csv"
edges.to_csv(EDGE_PATH, index=False)

print("\nSaved network edges to:", EDGE_PATH.resolve())


Total segments: 175002
Zero-distance segments: 302
Segments over 300 km: 530
Missing segment distances: 0
Duplicate train segments: 0


,Train No,Station Code,Next Station Code,Segment Distance
95,557,URK,PJP,1301.0
98,604,MGS,ASR,1222.0
90,504,MGS,BTI,1106.0
93,506,MGS,PTA,954.0
36,290,UDZ,JSM,839.0
59,421,BE,SVDK,820.0
61,422,SVDK,BE,820.0
11810,12293,BSL,STA,741.0
11814,12294,STA,BSL,741.0
11113,12214,DEE,HBJ,696.0



Saved network edges to: C:\Users\Harsh\Desktop\kalantar\data\processed\railway_network_edges.csv


In [9]:

# Inspect zero-distance segments
print("ZERO-DISTANCE SEGMENTS")
display(
    edges.loc[
        edges["Segment Distance"] == 0,
        ["Train No", "Station Code", "Next Station Code",
         "Distance", "Next Distance"]
    ].head(15)
)

# Inspect unusually long segments
print("\nSEGMENTS OVER 300 KM")
display(
    edges.loc[
        edges["Segment Distance"] > 300,
        ["Train No", "Station Code", "Next Station Code",
         "Distance", "Next Distance", "Segment Distance"]
    ].sort_values("Segment Distance", ascending=False).head(15)
)

# Inspect one affected train's full route
long_segment = edges.loc[
    edges["Segment Distance"] > 300
].sort_values("Segment Distance", ascending=False).iloc[0]

train_id = long_segment["Train No"]

print(f"\nFull route for train {train_id}:")
display(
    routes.loc[
        routes["Train No"] == train_id,
        ["Train No", "SEQ", "Station Code",
         "Station Name", "Distance"]
    ].sort_values("SEQ")
)


ZERO-DISTANCE SEGMENTS


,Train No,Station Code,Next Station Code,Distance,Next Distance
30534,14151,MNQ,SKB,196.0,196.0
32132,14546,BTU,SUJR,106.0,106.0
43125,16515,KIGL,YY,136.0,136.0
43160,16516,BGNR,YY,513.0,513.0
43161,16516,YY,KIGL,513.0,513.0
43759,16575,KIGL,YY,136.0,136.0
43778,16576,BGNR,YY,258.0,258.0
43779,16576,YY,KIGL,258.0,258.0
49024,18007,JRG,BTQ,229.0,229.0
49032,18008,BTQ,JRG,31.0,31.0



SEGMENTS OVER 300 KM


,Train No,Station Code,Next Station Code,Distance,Next Distance,Segment Distance
95,557,URK,PJP,0.0,1301.0,1301.0
98,604,MGS,ASR,207.0,1429.0,1222.0
90,504,MGS,BTI,207.0,1313.0,1106.0
93,506,MGS,PTA,207.0,1161.0,954.0
36,290,UDZ,JSM,821.0,1660.0,839.0
61,422,SVDK,BE,0.0,820.0,820.0
59,421,BE,SVDK,456.0,1276.0,820.0
11814,12294,STA,BSL,183.0,924.0,741.0
11810,12293,BSL,STA,428.0,1169.0,741.0
11113,12214,DEE,HBJ,0.0,696.0,696.0



Full route for train 557:


,Train No,SEQ,Station Code,Station Name,Distance
95,557,1.0,URK,URKURA,0.0
96,557,2.0,PJP,PANJIPARA,1301.0


In [10]:

# Number of stations recorded per train
station_counts = routes.groupby("Train No").size()

print("Trains with only 2 station records:",
      (station_counts == 2).sum())

print("Trains with fewer than 3 station records:",
      (station_counts < 3).sum())

# Summarize distance jumps
segment_summary = edges["Segment Distance"].describe(
    percentiles=[0.50, 0.90, 0.95, 0.99]
)
print("\nSegment distance summary:")
print(segment_summary)

# Count large jumps using several thresholds
for threshold in [100, 200, 300, 500, 1000]:
    count = (edges["Segment Distance"] > threshold).sum()
    print(f"Segments over {threshold} km: {count}")

# Inspect the complete routes for the largest jumps
largest = edges.nlargest(5, "Segment Distance")

for _, edge in largest.iterrows():
    train_id = edge["Train No"]
    print(f"\nTrain {train_id}:")
    display(
        routes.loc[
            routes["Train No"] == train_id,
            ["Train No", "SEQ", "Station Code",
             "Station Name", "Distance"]
        ].sort_values("SEQ")
    )


Trains with only 2 station records: 1249
Trains with fewer than 3 station records: 1249

Segment distance summary:
count    175002.000000
mean         22.160467
std          40.523841
min           0.000000
50%           8.000000
90%          59.000000
95%          91.000000
99%         199.000000
max        1301.000000
Name: Segment Distance, dtype: float64
Segments over 100 km: 7564
Segments over 200 km: 1723
Segments over 300 km: 530
Segments over 500 km: 36
Segments over 1000 km: 3

Train 557:


,Train No,SEQ,Station Code,Station Name,Distance
95,557,1.0,URK,URKURA,0.0
96,557,2.0,PJP,PANJIPARA,1301.0



Train 604:


,Train No,SEQ,Station Code,Station Name,Distance
97,604,1.0,PNBE,PATNA JN.,0.0
98,604,2.0,MGS,MUGHAL SARAI,207.0
99,604,3.0,ASR,AMRITSAR JN.,1429.0



Train 504:


,Train No,SEQ,Station Code,Station Name,Distance
89,504,1.0,PNBE,PATNA JN.,0.0
90,504,2.0,MGS,MUGHAL SARAI,207.0
91,504,3.0,BTI,BATHINDA JN,1313.0



Train 506:


,Train No,SEQ,Station Code,Station Name,Distance
92,506,1.0,PNBE,PATNA JN.,0.0
93,506,2.0,MGS,MUGHAL SARAI,207.0
94,506,3.0,PTA,PATIALA,1161.0



Train 290:


,Train No,SEQ,Station Code,Station Name,Distance
30,290,1.0,DSJ,DELHI-SAFDAR,0.0
31,290,2.0,GADJ,GANDHINAGAR,306.0
32,290,3.0,JP,JAIPUR JN.,311.0
33,290,4.0,DPA,DURGAPURA,318.0
34,290,5.0,SWM,SAWAI MADHOP,442.0
35,290,6.0,COR,CHITTAURGARH,707.0
36,290,7.0,UDZ,UDAIPUR CITY,821.0
37,290,8.0,JSM,JAISALMER,1660.0
38,290,9.0,MDB,MANDOR,1951.0
39,290,10.0,FL,PHULERA JN.,2213.0


In [11]:

# Check whether cumulative distance decreases within a train
distance_changes = (
    routes.groupby("Train No")["Distance"].diff()
)

print(
    "Negative cumulative-distance changes:",
    (distance_changes < 0).sum()
)

# Inspect all segments over 300 km
suspicious = edges.loc[
    edges["Segment Distance"] > 300,
    [
        "Train No", "Station Code", "Next Station Code",
        "Distance", "Next Distance", "Segment Distance"
    ]
].copy()

print("Segments flagged for review:", len(suspicious))
display(suspicious.head(20))

# Count how many trains contain at least one flagged segment
print(
    "Trains with a segment over 300 km:",
    suspicious["Train No"].nunique()
)


Negative cumulative-distance changes: 0
Segments flagged for review: 530


,Train No,Station Code,Next Station Code,Distance,Next Distance,Segment Distance
5926,11055,BSL,ET,428.0,735.0,307.0
5962,11056,ET,BSL,997.0,1304.0,307.0
6151,11059,KYN,BSL,37.0,428.0,391.0
6152,11059,BSL,ET,428.0,735.0,307.0
6192,11060,ET,BSL,1035.0,1342.0,307.0
6193,11060,BSL,KYN,1342.0,1732.0,390.0
9278,12101,KYN,BSL,37.0,428.0,391.0
9306,12102,BSL,IGP,1534.0,1842.0,308.0
9393,12107,BSL,BPL,428.0,826.0,398.0
9405,12108,BPL,BSL,581.0,979.0,398.0


Trains with a segment over 300 km: 337


In [12]:

network = edges.copy()

# Flag unusual distances for later validation
network["distance_review"] = (
    network["Segment Distance"] > 300
)

network["zero_distance"] = (
    network["Segment Distance"] == 0
)

# Only mark distance as usable when it is non-negative
# and not currently flagged for review.
network["distance_usable"] = (
    network["Segment Distance"].ge(0)
    & ~network["distance_review"]
    & ~network["zero_distance"]
)

# Save the full network with quality flags
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NETWORK_PATH = OUTPUT_DIR / "railway_network_edges_flagged.csv"
network.to_csv(NETWORK_PATH, index=False)

print("Total edges:", len(network))
print("Distance flagged for review:", network["distance_review"].sum())
print("Zero-distance edges:", network["zero_distance"].sum())
print("Edges with usable distance:", network["distance_usable"].sum())
print("Saved to:", NETWORK_PATH.resolve())


Total edges: 175002
Distance flagged for review: 530
Zero-distance edges: 302
Edges with usable distance: 174170
Saved to: C:\Users\Harsh\Desktop\kalantar\data\processed\railway_network_edges_flagged.csv
